# Robust Human Perception — Final Evaluation Notebook

This notebook documents the completed final evaluation workflow.

It deliberately starts from a **fresh extraction** and a **new empty result table**. It does not reuse failed local CSVs or run smoke tests that append rows to final results.

It also:

- repairs Windows paths before evaluation;
- uses stable unsharp masking for motion-blur enhancement;
- removes duplicate rows automatically after every stage;
- verifies exact expected row counts after every stage;
- rebuilds the aggregate table from the final clean per-image table;
- reports detection F1 and recall;
- backs up each completed stage to Google Drive.

## Required files in the root of Google Drive

- `colab_eval_package.zip`
- `pose_robust_ft_best.pt`

Select a GPU runtime and run every cell from top to bottom exactly once.


In [5]:
# 1. Mount Google Drive and define paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil
import os
import sys
import subprocess
import json
import re
import yaml
import pandas as pd
import numpy as np

DRIVE = Path("/content/drive/MyDrive")
EVAL_ZIP = DRIVE / "colab_eval_package.zip"
BEST_PT = DRIVE / "pose_robust_ft_best.pt"

ROOT = Path("/content/robust_human_eval")
RESULTS_ROOT = ROOT / "results" / "coco_person"
TABLES_DIR = RESULTS_ROOT / "tables"
PER_IMAGE_CSV = TABLES_DIR / "val_per_image.csv"
AGGREGATE_CSV = TABLES_DIR / "val_aggregate.csv"

DRIVE_BACKUP = DRIVE / "robust_human_results_fixed"
FINAL_ZIP_BASE = DRIVE / "robust_human_final_results_fixed"

FINAL_N = 150
IMG_SIZE = 416

assert EVAL_ZIP.exists(), f"Missing Google Drive file: {EVAL_ZIP}"
assert BEST_PT.exists(), f"Missing Google Drive file: {BEST_PT}"

DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

print("Evaluation package:", EVAL_ZIP)
print("Fine-tuned checkpoint:", BEST_PT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Evaluation package: /content/drive/MyDrive/colab_eval_package.zip
Fine-tuned checkpoint: /content/drive/MyDrive/pose_robust_ft_best.pt


In [6]:
# 2. Verify GPU and install dependencies
import torch

assert torch.cuda.is_available(), (
    "GPU is not enabled. In Colab choose Runtime → Change runtime type → GPU."
)
print("GPU:", torch.cuda.get_device_name(0))

%pip install -q ultralytics opencv-python-headless pycocotools PyYAML pandas matplotlib scipy scikit-image tqdm requests


GPU: NVIDIA A100-SXM4-40GB


In [7]:
# 3. Extract a completely fresh project

# This intentionally deletes the old Colab project and its corrupted CSV.
if ROOT.exists():
    shutil.rmtree(ROOT)

ROOT.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(str(EVAL_ZIP), str(ROOT))

assert (ROOT / "scripts").exists(), "The evaluation ZIP has an unexpected structure."
assert (ROOT / "src").exists(), "The evaluation ZIP has an unexpected structure."

print("Fresh project extracted to:", ROOT)
print("Top-level entries:", sorted(p.name for p in ROOT.iterdir()))


Fresh project extracted to: /content/robust_human_eval
Top-level entries: ['configs', 'data', 'requirements.txt', 'results', 'scripts', 'src']


In [8]:
# 4. Install project requirements and copy the trained checkpoint
requirements = ROOT / "requirements.txt"
if requirements.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

local_best = RESULTS_ROOT / "checkpoints" / "pose_robust_ft_best.pt"
local_best.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST_PT, local_best)

assert local_best.exists() and local_best.stat().st_size > 0
print("Checkpoint copied to:", local_best)


Checkpoint copied to: /content/robust_human_eval/results/coco_person/checkpoints/pose_robust_ft_best.pt


In [9]:
# 5. Repair all Windows paths robustly

from pathlib import Path, PureWindowsPath
import json
import re
import yaml

TEXT_EXTENSIONS = {".yaml", ".yml", ".json", ".csv", ".txt", ".py"}

def convert_windows_project_path(value):
    """
    Convert any string containing the old Windows project root into
    an equivalent path under /content/robust_human_eval.

    Non-path strings are returned unchanged.
    """
    if not isinstance(value, str):
        return value

    normalized = value.replace("\\", "/")

    # Handle malformed paths created by earlier replacement attempts.
    normalized = normalized.replace(
        "C: /content/robust_human_eval",
        str(ROOT),
    )
    normalized = normalized.replace(
        "C:/content/robust_human_eval",
        str(ROOT),
    )

    marker = "Robust Human Detection"
    lowered = normalized.lower()

    if marker.lower() in lowered:
        marker_index = lowered.index(marker.lower())
        suffix_start = marker_index + len(marker)
        suffix = normalized[suffix_start:].lstrip("/")

        if suffix:
            return str(ROOT / Path(suffix))
        return str(ROOT)

    return normalized if normalized != value and "C:/Users/" in normalized else value


def repair_nested_object(value):
    """Recursively repair path strings inside dicts and lists."""
    if isinstance(value, dict):
        return {
            key: repair_nested_object(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [repair_nested_object(item) for item in value]

    if isinstance(value, str):
        return convert_windows_project_path(value)

    return value


changed_files = []

for path in ROOT.rglob("*"):
    if not path.is_file():
        continue

    suffix = path.suffix.lower()
    if suffix not in TEXT_EXTENSIONS:
        continue

    try:
        if suffix == ".json":
            original_object = json.loads(
                path.read_text(encoding="utf-8")
            )
            repaired_object = repair_nested_object(original_object)

            if repaired_object != original_object:
                path.write_text(
                    json.dumps(
                        repaired_object,
                        ensure_ascii=False,
                        indent=2,
                    ),
                    encoding="utf-8",
                )
                changed_files.append(path)

        elif suffix in {".yaml", ".yml"}:
            original_object = yaml.safe_load(
                path.read_text(encoding="utf-8")
            )
            repaired_object = repair_nested_object(original_object)

            if repaired_object != original_object:
                path.write_text(
                    yaml.safe_dump(
                        repaired_object,
                        sort_keys=False,
                        allow_unicode=True,
                    ),
                    encoding="utf-8",
                )
                changed_files.append(path)

        else:
            old_text = path.read_text(
                encoding="utf-8",
                errors="ignore",
            )

            # Replace any complete old Windows project prefix while
            # preserving the path suffix after the project directory.
            pattern = re.compile(
                r"C:[/\\]Users[/\\][^\n\r\"']*?"
                r"Robust Human Detection",
                flags=re.IGNORECASE,
            )

            new_text = pattern.sub(str(ROOT), old_text)
            new_text = new_text.replace(
                "C: /content/robust_human_eval",
                str(ROOT),
            )
            new_text = new_text.replace(
                "C:/content/robust_human_eval",
                str(ROOT),
            )

            if new_text != old_text:
                path.write_text(new_text, encoding="utf-8")
                changed_files.append(path)

    except Exception as exc:
        raise RuntimeError(
            f"Failed while repairing paths in {path}: {exc}"
        ) from exc


# Explicitly overwrite the split metadata used by the dataset loader.
split_yaml = ROOT / "data" / "splits" / "coco_person_pose.yaml"
assert split_yaml.exists(), f"Missing split file: {split_yaml}"

split_cfg = yaml.safe_load(
    split_yaml.read_text(encoding="utf-8")
) or {}

if not isinstance(split_cfg.get("val"), dict):
    split_cfg["val"] = {}

split_cfg["val"]["ann_file"] = str(
    ROOT
    / "data"
    / "raw"
    / "coco"
    / "annotations"
    / "person_keypoints_val2017.json"
)
split_cfg["val"]["image_dir"] = str(
    ROOT / "data" / "selected" / "val2017"
)
split_cfg["val"]["ids_file"] = str(
    ROOT / "data" / "splits" / "val_person_ids.txt"
)

if not isinstance(split_cfg.get("train"), dict):
    split_cfg["train"] = {}

split_cfg["train"]["ann_file"] = str(
    ROOT
    / "data"
    / "raw"
    / "coco"
    / "annotations"
    / "person_keypoints_train2017.json"
)
split_cfg["train"]["image_dir"] = str(
    ROOT / "data" / "selected" / "train2017"
)
split_cfg["train"]["ids_file"] = str(
    ROOT / "data" / "splits" / "train_person_ids.txt"
)

if "path" in split_cfg:
    split_cfg["path"] = str(
        ROOT / "data" / "splits" / "yolo_pose"
    )

split_yaml.write_text(
    yaml.safe_dump(
        split_cfg,
        sort_keys=False,
        allow_unicode=True,
    ),
    encoding="utf-8",
)


# Explicitly repair train_ids.json and val_ids.json if present.
for ids_json in [
    ROOT / "data" / "splits" / "train_ids.json",
    ROOT / "data" / "splits" / "val_ids.json",
]:
    if ids_json.exists():
        ids_object = json.loads(
            ids_json.read_text(encoding="utf-8")
        )
        repaired_ids = repair_nested_object(ids_object)
        ids_json.write_text(
            json.dumps(
                repaired_ids,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )


# Verify no Windows paths remain.
remaining_windows_paths = []

for path in ROOT.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in TEXT_EXTENSIONS:
        continue

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore",
        )
    except Exception:
        continue

    windows_markers = [
        "C:/Users/",
        "C:\\Users\\",
        "C: /content/robust_human_eval",
        "C:/content/robust_human_eval",
    ]

    if any(marker in text for marker in windows_markers):
        remaining_windows_paths.append(path)

assert not remaining_windows_paths, (
    "Windows paths remain in: "
    + ", ".join(map(str, remaining_windows_paths))
)


required_paths = {
    "validation annotations": Path(
        split_cfg["val"]["ann_file"]
    ),
    "validation image directory": Path(
        split_cfg["val"]["image_dir"]
    ),
    "validation ID list": Path(
        split_cfg["val"]["ids_file"]
    ),
    "fine-tuned checkpoint": local_best,
}

for name, path in required_paths.items():
    print(
        f"{name}: {path} | exists={path.exists()}"
    )
    assert path.exists(), f"Missing {name}: {path}"

validation_images = [
    path
    for path in required_paths[
        "validation image directory"
    ].iterdir()
    if path.is_file()
]

assert len(validation_images) >= FINAL_N, (
    f"Expected at least {FINAL_N} validation images; "
    f"found {len(validation_images)}."
)

print("Changed files:")
for path in changed_files:
    print(" -", path)

print("Path repair passed.")


validation annotations: /content/robust_human_eval/data/raw/coco/annotations/person_keypoints_val2017.json | exists=True
validation image directory: /content/robust_human_eval/data/selected/val2017 | exists=True
validation ID list: /content/robust_human_eval/data/splits/val_person_ids.txt | exists=True
fine-tuned checkpoint: /content/robust_human_eval/results/coco_person/checkpoints/pose_robust_ft_best.pt | exists=True
Changed files:
 - /content/robust_human_eval/data/splits/coco_person_pose.yaml
 - /content/robust_human_eval/data/splits/train_ids.json
 - /content/robust_human_eval/data/splits/val_ids.json
Path repair passed.


In [10]:
# 6. Replace unstable motion-blur enhancement with a stable implementation

stable_file = ROOT / "src" / "enhancement" / "stable_motion_blur.py"
stable_file.parent.mkdir(parents=True, exist_ok=True)

stable_file.write_text(
    '''import cv2
import numpy as np


def enhance_motion_blur(
    image: np.ndarray,
    sigma: float = 1.2,
    amount: float = 0.6,
    **kwargs,
) -> np.ndarray:
    """Stable unsharp-mask enhancement for motion-blurred images."""
    if image is None:
        raise ValueError("Input image is None.")

    source = np.asarray(image)
    if source.dtype != np.uint8:
        source = np.clip(source, 0, 255).astype(np.uint8)

    smooth = cv2.GaussianBlur(
        source,
        ksize=(0, 0),
        sigmaX=float(sigma),
        sigmaY=float(sigma),
    )

    enhanced = cv2.addWeighted(
        source,
        1.0 + float(amount),
        smooth,
        -float(amount),
        0,
    )

    return np.clip(enhanced, 0, 255).astype(np.uint8)
''',
    encoding="utf-8",
)

registry_file = ROOT / "src" / "enhancement" / "registry.py"

registry_file.write_text(
    '''"""Enhancement registry matched to distortion families."""
from __future__ import annotations

from typing import Any

import numpy as np

from src.common.io import load_yaml, project_path
from src.enhancement.jpeg import enhance_jpeg
from src.enhancement.low_light import enhance_low_light
from src.enhancement.stable_motion_blur import enhance_motion_blur


def default_cfg() -> dict[str, Any]:
    return load_yaml(project_path("configs", "distortions.yaml"))


def apply_enhancement(
    image: np.ndarray,
    distortion_name: str,
    distortion_meta: dict[str, Any] | None = None,
    cfg: dict[str, Any] | None = None,
) -> tuple[np.ndarray, dict[str, Any]]:
    full = cfg or default_cfg()
    enh = full["enhancement"]
    distortion_meta = distortion_meta or {}

    if distortion_name == "low_light":
        c = enh["low_light"]
        return enhance_low_light(
            image,
            clahe_clip_limit=float(c["clahe_clip_limit"]),
            clahe_tile_size=int(c["clahe_tile_size"]),
            brighten_gamma=float(c["brighten_gamma"]),
        )

    if distortion_name == "motion_blur":
        # Stable conservative unsharp-mask parameters used in the final run.
        sigma = 1.2
        amount = 0.6

        enhanced = enhance_motion_blur(
            image,
            sigma=sigma,
            amount=amount,
        )

        metadata = {
            "enhancement_method": "stable_unsharp_mask",
            "unsharp_sigma": sigma,
            "unsharp_amount": amount,
            "source_kernel_length": int(
                distortion_meta.get("kernel_length", 15)
            ),
            "source_angle": float(
                distortion_meta.get("angle", 0.0)
            ),
        }
        return enhanced, metadata

    if distortion_name == "jpeg":
        c = enh["jpeg"]
        return enhance_jpeg(
            image,
            bilateral_d=int(c["bilateral_d"]),
            bilateral_sigma_color=float(c["bilateral_sigma_color"]),
            bilateral_sigma_space=float(c["bilateral_sigma_space"]),
        )

    raise ValueError(f"No enhancement for distortion: {distortion_name}")
''',
    encoding="utf-8",
)

# Compile both generated files now so syntax errors are caught before evaluation.
compile(stable_file.read_text(encoding="utf-8"), str(stable_file), "exec")
compile(registry_file.read_text(encoding="utf-8"), str(registry_file), "exec")

print("Stable motion-blur enhancement installed.")


Stable motion-blur enhancement installed.


In [11]:
# 7. Preflight-test the enhancement registry without writing result rows
%cd /content/robust_human_eval

import importlib
import cv2

for module_name in [
    "src.enhancement.registry",
    "src.enhancement.stable_motion_blur",
]:
    sys.modules.pop(module_name, None)

registry = importlib.import_module("src.enhancement.registry")

sample_image_path = next((ROOT / "data" / "selected" / "val2017").glob("*.jpg"))
sample_image = cv2.imread(str(sample_image_path))
assert sample_image is not None

enhanced_image, enhancement_metadata = registry.apply_enhancement(
    image=sample_image,
    distortion_name="motion_blur",
    distortion_meta={"kernel_length": 15, "angle": 0.0},
)

assert enhanced_image.shape == sample_image.shape
assert enhanced_image.dtype == np.uint8
assert enhancement_metadata["enhancement_method"] == "stable_unsharp_mask"

print("Registry preflight passed.")
print("Metadata:", enhancement_metadata)


/content/robust_human_eval
Registry preflight passed.
Metadata: {'enhancement_method': 'stable_unsharp_mask', 'unsharp_sigma': 1.2, 'unsharp_amount': 0.6, 'source_kernel_length': 15, 'source_angle': 0.0}


In [12]:
# 8. Create a brand-new empty final-results area

# Preserve only the checkpoint. Remove all tables, figures, logs, and qualitative files
# that may have been included in the ZIP or created by earlier runs.
for folder_name in ["tables", "figures", "logs", "qualitative"]:
    folder = RESULTS_ROOT / folder_name
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

assert not PER_IMAGE_CSV.exists()

print("Final result area is clean.")


Final result area is clean.


In [13]:
# 9. Helper functions for safe execution, deduplication, validation, and backup

UNIQUE_KEY = [
    "dataset_name",
    "dataset_split",
    "image_id",
    "task",
    "condition",
    "model_name",
    "weights_path",
]

def run_command(arguments: list[str]) -> None:
    print("Running:", " ".join(arguments))
    subprocess.run(
        arguments,
        cwd=str(ROOT),
        check=True,
        env={**os.environ, "PYTHONPATH": str(ROOT)},
    )

def load_results() -> pd.DataFrame:
    assert PER_IMAGE_CSV.exists(), f"Missing result table: {PER_IMAGE_CSV}"
    return pd.read_csv(PER_IMAGE_CSV, low_memory=False)

def normalize_and_deduplicate_results() -> pd.DataFrame:
    df = load_results()

    missing_keys = [column for column in UNIQUE_KEY if column not in df.columns]
    assert not missing_keys, f"Missing unique-key columns: {missing_keys}"

    # Canonicalize nullable path/name fields before duplicate comparison.
    for column in ["dataset_name", "dataset_split", "task", "condition", "model_name", "weights_path"]:
        df[column] = df[column].fillna("").astype(str).str.replace("\\", "/", regex=False)

    if "timestamp" in df.columns:
        df["_timestamp_sort"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df = df.sort_values("_timestamp_sort", na_position="first")
        df = df.drop(columns="_timestamp_sort")

    df = (
        df.drop_duplicates(subset=UNIQUE_KEY, keep="last")
        .reset_index(drop=True)
    )

    df.to_csv(PER_IMAGE_CSV, index=False)
    return df

def assert_results(
    expected_rows: int,
    expected_groups: int,
    stage_name: str,
) -> pd.DataFrame:
    df = normalize_and_deduplicate_results()

    duplicate_count = int(
        df.duplicated(subset=UNIQUE_KEY, keep=False).sum()
    )

    success_text = df["success"].astype(str).str.lower()
    failed_count = int((~success_text.isin(["true", "1"])).sum())

    group_counts = (
        df.groupby(
            ["task", "model_name", "weights_path", "condition"],
            dropna=False,
        )
        .size()
        .reset_index(name="rows")
    )

    bad_groups = group_counts[group_counts["rows"] != FINAL_N]

    print(f"{stage_name}:")
    print("  rows:", len(df))
    print("  experiment groups:", len(group_counts))
    print("  duplicates:", duplicate_count)
    print("  failed rows:", failed_count)
    print("  groups not containing 150 images:", len(bad_groups))

    if not bad_groups.empty:
        display(bad_groups)

    assert len(df) == expected_rows, (
        f"{stage_name}: expected {expected_rows} rows, found {len(df)}."
    )
    assert len(group_counts) == expected_groups, (
        f"{stage_name}: expected {expected_groups} groups, found {len(group_counts)}."
    )
    assert duplicate_count == 0, f"{stage_name}: duplicate rows remain."
    assert failed_count == 0, f"{stage_name}: failed result rows found."
    assert bad_groups.empty, f"{stage_name}: incomplete experiment groups found."

    return df

def backup_stage(name: str) -> None:
    destination = DRIVE_BACKUP / name
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(RESULTS_ROOT, destination)
    print("Backed up:", destination)

print("Safety helpers ready.")


Safety helpers ready.


## Full evaluation

The following three stages write to a fresh table.

There are no smoke-test rows and no `--resume` option. After each stage the notebook deduplicates defensively, verifies exact counts, and backs up the completed results to Google Drive.


In [14]:
# 10. Stage 1 — Clean baseline

run_command([
    sys.executable,
    "scripts/01_run_clean.py",
    "--device", "0",
    "--max-images", str(FINAL_N),
    "--imgsz", str(IMG_SIZE),
])

clean_df = assert_results(
    expected_rows=450,
    expected_groups=3,
    stage_name="Clean baseline",
)

backup_stage("01_after_clean")


Running: /usr/bin/python3 scripts/01_run_clean.py --device 0 --max-images 150 --imgsz 416
Clean baseline:
  rows: 450
  experiment groups: 3
  duplicates: 0
  failed rows: 0
  groups not containing 150 images: 0
Backed up: /content/drive/MyDrive/robust_human_results_fixed/01_after_clean


In [15]:
# 11. Stage 2 — Original models on distorted and enhanced images

run_command([
    sys.executable,
    "scripts/02_run_distort_enhance.py",
    "--device", "0",
    "--max-images", str(FINAL_N),
    "--imgsz", str(IMG_SIZE),
])

distorted_df = assert_results(
    expected_rows=8550,
    expected_groups=57,
    stage_name="Original distorted/enhanced evaluation",
)

backup_stage("02_after_distort_enhance")


Running: /usr/bin/python3 scripts/02_run_distort_enhance.py --device 0 --max-images 150 --imgsz 416
Original distorted/enhanced evaluation:
  rows: 8550
  experiment groups: 57
  duplicates: 0
  failed rows: 0
  groups not containing 150 images: 0
Backed up: /content/drive/MyDrive/robust_human_results_fixed/02_after_distort_enhance


In [16]:
# 12. Stage 3 — Fine-tuned pose model

run_command([
    sys.executable,
    "scripts/05_eval_finetuned.py",
    "--weights", str(local_best),
    "--device", "0",
    "--max-images", str(FINAL_N),
    "--imgsz", str(IMG_SIZE),
])

final_df = assert_results(
    expected_rows=11400,
    expected_groups=76,
    stage_name="Complete evaluation",
)

backup_stage("03_complete_raw_results")


Running: /usr/bin/python3 scripts/05_eval_finetuned.py --weights /content/robust_human_eval/results/coco_person/checkpoints/pose_robust_ft_best.pt --device 0 --max-images 150 --imgsz 416
Complete evaluation:
  rows: 11400
  experiment groups: 76
  duplicates: 0
  failed rows: 0
  groups not containing 150 images: 0
Backed up: /content/drive/MyDrive/robust_human_results_fixed/03_complete_raw_results


In [17]:
# 13. Build a fresh aggregate table from the verified 11,400-row table

df = load_results()

group_columns = [
    "task",
    "condition",
    "distortion",
    "severity",
    "enhanced",
    "enhancement_method",
    "model_name",
    "weights_path",
    "device",
]

metric_columns = [
    "psnr",
    "mse",
    "boundary_precision",
    "boundary_recall",
    "boundary_f1",
    "mean_edge_distance",
    "num_gt",
    "num_predictions",
    "true_positives",
    "false_positives",
    "false_negatives",
    "precision",
    "recall",
    "f1",
    "miss_rate",
    "mean_confidence",
    "num_gt_people",
    "num_matched_people",
    "pck_02",
    "pck_05",
    "normalized_keypoint_error",
    "oks",
]

group_columns = [column for column in group_columns if column in df.columns]
metric_columns = [column for column in metric_columns if column in df.columns]

for column in metric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

means = (
    df.groupby(group_columns, dropna=False, as_index=False)[metric_columns]
    .mean()
)

counts = (
    df.groupby(group_columns, dropna=False)
    .size()
    .reset_index(name="n_images")
)

aggregate = means.merge(
    counts,
    on=group_columns,
    how="left",
)

aggregate.to_csv(AGGREGATE_CSV, index=False)

print("Aggregate rows:", len(aggregate))
print("Image counts:")
print(aggregate["n_images"].value_counts().sort_index())

assert len(aggregate) == 76, (
    f"Expected 76 aggregate rows; found {len(aggregate)}."
)
assert aggregate["n_images"].eq(FINAL_N).all(), (
    "Every aggregate row must contain exactly 150 images."
)

print("Aggregate table verified:", AGGREGATE_CSV)


Aggregate rows: 76
Image counts:
n_images
150    76
Name: count, dtype: int64
Aggregate table verified: /content/robust_human_eval/results/coco_person/tables/val_aggregate.csv


In [18]:
# 14. Validate with the repository validator and render final plots

run_command([
    sys.executable,
    "scripts/08_validate_results.py",
])

# Plot labels and filenames are defined correctly in the committed source.
# Do not patch repository code at runtime.
figures_dir = RESULTS_ROOT / "figures"
if figures_dir.exists():
    shutil.rmtree(figures_dir)
figures_dir.mkdir(parents=True, exist_ok=True)

run_command([
    sys.executable,
    "scripts/06_make_plots.py",
])

print("Generated figures:")
for path in sorted(figures_dir.iterdir()):
    if path.is_file():
        print(" -", path.name)


Running: /usr/bin/python3 scripts/08_validate_results.py
Running: /usr/bin/python3 scripts/06_make_plots.py
Generated figures:
 - boundary_f1_vs_severity.csv
 - boundary_f1_vs_severity.pdf
 - boundary_f1_vs_severity.png
 - boundary_vs_detection.pdf
 - boundary_vs_detection.png
 - det_f1_vs_severity.csv
 - det_f1_vs_severity.pdf
 - det_f1_vs_severity.png
 - det_recall_vs_severity.csv
 - det_recall_vs_severity.pdf
 - det_recall_vs_severity.png
 - pose_pck_vs_psnr.pdf
 - pose_pck_vs_psnr.png
 - pose_pck_vs_severity.csv
 - pose_pck_vs_severity.pdf
 - pose_pck_vs_severity.png


In [19]:
# 15. Generate qualitative examples

qualitative_dir = RESULTS_ROOT / "qualitative"
if qualitative_dir.exists():
    shutil.rmtree(qualitative_dir)
qualitative_dir.mkdir(parents=True, exist_ok=True)

run_command([
    sys.executable,
    "scripts/07_visualize_samples.py",
    "--severity", "medium",
    "--device", "0",
])

print("Generated qualitative files:")
for path in sorted(qualitative_dir.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(RESULTS_ROOT))


Running: /usr/bin/python3 scripts/07_visualize_samples.py --severity medium --device 0
Generated qualitative files:
 - qualitative/qualitative_meta.json
 - qualitative/sample_2473_medium.png


In [20]:
# 16. Final verification, persistent backup, and ZIP

final_df = assert_results(
    expected_rows=11400,
    expected_groups=76,
    stage_name="Final verification",
)

aggregate = pd.read_csv(AGGREGATE_CSV, low_memory=False)
assert len(aggregate) == 76
assert aggregate["n_images"].eq(FINAL_N).all()

final_folder = DRIVE_BACKUP / "final"
if final_folder.exists():
    shutil.rmtree(final_folder)
shutil.copytree(RESULTS_ROOT, final_folder)

zip_file = Path(
    shutil.make_archive(
        str(FINAL_ZIP_BASE),
        "zip",
        RESULTS_ROOT,
    )
)

assert zip_file.exists() and zip_file.stat().st_size > 0

print("FINAL SUCCESS")
print("Per-image rows:", len(final_df))
print("Aggregate rows:", len(aggregate))
print("Final Drive folder:", final_folder)
print("Final ZIP:", zip_file)
print("ZIP size MB:", round(zip_file.stat().st_size / 1024**2, 2))


Final verification:
  rows: 11400
  experiment groups: 76
  duplicates: 0
  failed rows: 0
  groups not containing 150 images: 0
FINAL SUCCESS
Per-image rows: 11400
Aggregate rows: 76
Final Drive folder: /content/drive/MyDrive/robust_human_results_fixed/final
Final ZIP: /content/drive/MyDrive/robust_human_final_results_fixed.zip
ZIP size MB: 8.69


# Finished

The final file is saved in Google Drive as:

`robust_human_final_results_fixed.zip`

Do not rerun individual evaluation cells after completion. To repeat the experiment, restart the runtime and run this notebook from the beginning so it creates a fresh result table.


In [30]:
print("")